<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_2%EC%A1%B0_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 소설 작가 분류 AI 경진대회

- 본 알고리즘은 작가별 문체 차이를 반영하기 위해 단어 기반 TF-IDF, 문자 기반 TF-IDF, 문장 구조 기반 수치 특징을 함께 사용
- 단어 TF-IDF는 작가가 자주 사용하는 어휘와 표현을 포착하고, 문자 TF-IDF는 접미사, 축약형, 구두점 사용 등 세밀한 문체 패턴을 반영
- 또한 문장 길이, 평균 단어 길이, 쉼표·마침표·따옴표 사용 빈도 등의 수치 특징을 추가하여 문체적 습관을 보완적으로 학습
- 최종적으로 이 특징들을 결합한 뒤 모델을 학습하여 각 문장이 어느 작가의 문체에 가까운지 확률 형태로 예측

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [3]:
# 파일 경로 설정
sub_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/sample_submission.csv'
train_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/train.csv'
test_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/test_x.csv'

# 데이터 로드
X_train = pd.read_csv(train_path)
X_test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

In [5]:
Y_train = LabelEncoder().fit_transform(X_train['author'])
Y_train

array([3, 2, 1, ..., 1, 3, 0])

In [6]:
X_train['author'].value_counts()

,count
author,
3,15063
0,13235
2,11554
4,7805
1,7222


- authors = ['0','1','2','3','4']

약간의 데이터 불균형

## preprocessing

In [7]:
import re

def clean(X_train,X_test):
    X_train['words'] = [re.sub("[^a-zA-Z]"," ", data).lower().split() for data in X_train['text']]
    X_test['words'] = [re.sub("[^a-zA-Z]"," ", data).lower().split() for data in X_test['text']]
    return X_train,X_test

In [8]:
X_train, X_test = clean(X_train, X_test)

- .split() : 공백 기준 단어 분리
- text의 특수문자 제거 > 소문자화 > 단어 분리

## feature engineering

### Punctuation

In [9]:
punctuations = [
    {"id":1,"p":"[;:]"},
    {"id":2,"p":"[,.]"},
    {"id":3,"p":"[?]"},
    {"id":4,"p":"[\']"},
    {"id":5,"p":"[\"]"},
    {"id":6,"p":"[;:,.?\'\"]"}
]

In [10]:
for p in punctuations:
    punctuation = p["p"]
    _train =  [ sentence.split() for sentence in X_train['text'] ]
    X_train['punc_'+str(p["id"])] = [len([
        word for word in sentence
        if bool(re.search(punctuation, word))])*100.0/(len(sentence)+1)  # 비율화
        for sentence in _train]

    _test =  [ sentence.split() for sentence in X_test['text'] ]
    X_test['punc_'+str(p["id"])] = [len([
        word for word in sentence
        if bool(re.search(punctuation, word))])*100.0/(len(sentence)+1)
        for sentence in _test]

- len([...]) : 구두점 포함 단어 개수 계산 = 해당 구두점 사용 횟수

비율로 계산해 문장길이 영향 적고, 문장길이+1 해주고 나눠줘 계산오류 방지

> 글쓰기 style 수치화

In [11]:
X_train.head()

,index,text,author,words,punc_1,punc_2,punc_3,punc_4,punc_5,punc_6
0,0,"He was almost choking. There was so much, so m...",3,"[he, was, almost, choking, there, was, so, muc...",2.127660,14.893617,0.0,0.000000,0.0,17.021277
1,1,"“Your sister asked for it, I suppose?”",2,"[your, sister, asked, for, it, i, suppose]",0.000000,12.500000,12.5,0.000000,0.0,25.000000
2,2,"She was engaged one day as she walked, in per...",1,"[she, was, engaged, one, day, as, she, walked,...",1.724138,13.793103,0.0,0.000000,0.0,15.517241
3,3,"The captain was in the porch, keeping himself ...",4,"[the, captain, was, in, the, porch, keeping, h...",3.389831,25.423729,0.0,1.694915,0.0,30.508475
4,4,"“Have mercy, gentlemen!” odin flung up his han...",3,"[have, mercy, gentlemen, odin, flung, up, his,...",2.500000,17.500000,0.0,0.000000,0.0,20.000000


### Stop Words

- 일반 NLP에서는 stopword를 제거
- 하지만 문제 분석이기에 stopword 제거 X
  - 작가마다 문체 스타일이 다르기 때문

In [12]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
stop_words = set(stopwords.words('english'))

X_train['stop_word'] = [
    len([word for word in sentence if word in stop_words])
    * 100.0 / (len(sentence)+1)

    for sentence in X_train['words']
]

X_test['stop_word'] = [
    len([word for word in sentence if word in stop_words])
    * 100.0 / (len(sentence)+1)

    for sentence in X_test['words']
]

### statistical style features

- 작가의 통계적 문체 특징 코드
- 긴 문장을 쓰는지, 다양한 단어를 쓰는지, 대문자를 많이 쓰는지, 대화체가 많은지 등

In [14]:
eng_stopwords = set(stopwords.words("english"))

In [15]:
import string

In [16]:
## Number of words in the text ##
X_train["num_words"] = X_train["text"].apply(lambda x: len(str(x).split()))
X_test["num_words"] = X_test["text"].apply(lambda x: len(str(x).split()))

## Number of unique words in the text ##
X_train["num_unique_words"] = X_train["text"].apply(lambda x: len(set(str(x).split())))
X_test["num_unique_words"] = X_test["text"].apply(lambda x: len(set(str(x).split())))

## Number of characters in the text ##
X_train["num_chars"] = X_train["text"].apply(lambda x: len(str(x)))  # 공백/구두점 포함 문자 개수
X_test["num_chars"] = X_test["text"].apply(lambda x: len(str(x)))

## Number of stopwords in the text ##
X_train["num_stopwords"] = X_train["text"].apply(lambda x: len([
    w for w in str(x).lower().split() if w in eng_stopwords
    ]))
X_test["num_stopwords"] = X_test["text"].apply(lambda x: len([
    w for w in str(x).lower().split() if w in eng_stopwords
    ]))

## Number of punctuations in the text ##
X_train["num_punctuations"] = X_train['text'].apply(lambda x: len([
    c for c in str(x) if c in string.punctuation
    ]))
X_test["num_punctuations"] = X_test['text'].apply(lambda x: len([
    c for c in str(x) if c in string.punctuation
    ]))

## Number of ALL CAPS words in the text ##
X_train["num_words_upper"] = X_train["text"].apply(lambda x: len([
    w for w in str(x).split() if w.isupper()  # 강조 스타일 반영 가능
    ]))
X_test["num_words_upper"] = X_test["text"].apply(lambda x: len([
    w for w in str(x).split() if w.isupper()
    ]))

## Number of Title Case words in the text(첫글자만 대문자) ##
X_train["num_words_title"] = X_train["text"].apply(lambda x: len([
    w for w in str(x).split() if w.istitle()
    ]))
X_test["num_words_title"] = X_test["text"].apply(lambda x: len([
    w for w in str(x).split() if w.istitle()
    ]))

## Average length of the words in the text ##
X_train["mean_word_len"] = X_train["text"].apply(lambda x: np.mean([
    len(w) for w in str(x).split()
    ]))
X_test["mean_word_len"] = X_test["text"].apply(lambda x: np.mean([
    len(w) for w in str(x).split()
    ]))

| feature          | 의미        |
| ---------------- | --------- |
| num_words        | 문장 길이     |
| num_unique_words | 어휘 다양성    |
| num_chars        | 전체 문자 길이  |
| num_stopwords    | 기능어 사용    |
| num_punctuations | 구두점 스타일   |
| num_words_upper  | 강조 스타일    |
| num_words_title  | 제목형 단어 사용 |
| mean_word_len    | 평균 단어 복잡도 |


불용어 쓰인 비율,개수를 피처화.  
NO!! 처럼 전부 대문자인 경우 / 고유면사, 캐릭터명 등 첫글자만 대문자인 경우 ...

### TF-IDF

**핵심 !!**

In [17]:
from sklearn import preprocessing, decomposition, model_selection, metrics, pipeline
from sklearn import ensemble, metrics, model_selection, naive_bayes

**< word - 단어단위 >**

In [48]:
def tfidfWords(X_train,X_test):
    tfidf_vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    full_tfidf = tfidf_vec.fit_transform(X_train['text'].values.tolist())
    train_tfidf = tfidf_vec.transform(X_train['text'].values.tolist())  # 학습 데이터의 TF-IDF 행렬
    test_tfidf = tfidf_vec.transform(X_test['text'].values.tolist())  # 테스트 데이터의 TF-IDF 행렬
    return train_tfidf,test_tfidf,full_tfidf

- 텍스트를 TF-IDF 숫자 벡터로 변환

In [19]:
def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)  # 각 작가일 확률 예측
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

- Multinomial Naive Bayes 학습
- 텍스트 분류에서 자주 쓰는 모델

In [20]:
def do_tfidf_MNB(X_train, X_test, Y_train):
    train_tfidf, test_tfidf, full_tfidf = tfidfWords(X_train, X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)  # train을 8개으로 나눔
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_tfidf[dev_index], train_tfidf[val_index]  # 학습용
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]  # 검증용
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_tfidf)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.  # 8개 모델의 예측 확률 평균
    return pred_train, pred_full_test

- OOF prediction = out-of-fold prediction
- dev_index, val_index in kf.split(X_train)
  - X_train 데이터를 8개의 fold로 나눈 뒤, 매 반복마다 학습용(dev) index, 검증용(val) index를 반환하는 것

In [46]:
#### 코드 추가 부분

from sklearn.linear_model import LogisticRegression

def do_tfidf_LR(X_train, X_test, Y_train, mode='words'):
    """
    로지스틱 회귀를 사용하여 8-Fold OOF 예측 확률 피처를 생성하는 함수
    mode: 'words' 또는 'chars' 선택에 따라 적절한 TF-IDF 함수를 호출합니다.
    """
    # 기존 팀원분의 TF-IDF 함수를 그대로 활용합니다.
    # 단, char 모드일 때는 함수명이 다를 수 있으니 기존 노트북의 함수명과 매칭 확인 필요!
    if mode == 'words':
        train_tfidf, test_tfidf, _ = tfidfWords(X_train, X_test) # 기존 단어 TF-IDF 함수
    else:
        # 기존 문자 TF-IDF 함수 (노트북에서 char 단위 tfidf를 수행하는 함수명으로 맞춤)
        # 만약 char 단위 함수명이 그냥 'tfidfWords'로 덮어씌워졌다면 그대로 사용
        train_tfidf, test_tfidf = tfidfWords(X_train, X_test)

    pred_train = np.zeros([X_train.shape[0], 5])
    pred_full_test = np.zeros([X_test.shape[0], 5])

    # 팀원분과 동일한 환경을 위해 8-Fold,일치하는 Random State 설정
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)
    cv_scores = []

    print(f"[{mode.upper()} - Logistic Regression] 8-Fold 메타 피처 생성 시작...")

    for fold, (dev_index, val_index) in enumerate(kf.split(X_train)):
        dev_X, val_X = train_tfidf[dev_index], train_tfidf[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]

        # 텍스트 분류에 최적화된 로지스틱 회귀 설정
        # C=1.0은 정규화 강도이며,만약 대용량 데이터에서 속도를 높이려면 solver='saga'도 좋습니다.
        model = LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', n_jobs=-1, random_state=42)
        model.fit(dev_X, dev_y)

        # 검증셋 및 테스트셋 예측 (확률값)
        pred_val_y = model.predict_proba(val_X)
        pred_test_y = model.predict_proba(test_tfidf)

        pred_train[val_index, :] = pred_val_y
        pred_full_test += pred_test_y

        fold_loss = metrics.log_loss(val_y, pred_val_y)
        cv_scores.append(fold_loss)
        print(f"  Fold {fold+1} LogLoss: {fold_loss:.4f}")

    print(f"⇒ Mean CV Score : {np.mean(cv_scores):.4f}\n")
    pred_full_test = pred_full_test / 8.

    return pred_train, pred_full_test

In [49]:
pred_train, pred_test = do_tfidf_MNB(X_train, X_test, Y_train)

Mean cv score :  1.0898864911765824


In [50]:
X_train["tfidf_words_nb_cvec_0"] = pred_train[:,0]
X_train["tfidf_words_nb_cvec_1"] = pred_train[:,1]
X_train["tfidf_words_nb_cvec_2"] = pred_train[:,2]
X_train["tfidf_words_nb_cvec_3"] = pred_train[:,3]
X_train["tfidf_words_nb_cvec_4"] = pred_train[:,4]
X_test["tfidf_words_nb_cvec_0"] = pred_test[:,0]
X_test["tfidf_words_nb_cvec_1"] = pred_test[:,1]
X_test["tfidf_words_nb_cvec_2"] = pred_test[:,2]
X_test["tfidf_words_nb_cvec_3"] = pred_test[:,3]
X_test["tfidf_words_nb_cvec_4"] = pred_test[:,4]

> TF-IDF + Naive Bayes로 1차 작가 단위 분류를 한 뒤, 그 예측 확률 5개를 새로운 feature로 추가하는 코드

**< char - 문자단위 >**

In [51]:
def tfidfWords(X_train,X_test):
    tfidf_vec = TfidfVectorizer(stop_words='english', ngram_range=(3,4), analyzer='char')
    full_tfidf = tfidf_vec.fit_transform(X_train['text'].values.tolist())
    train_tfidf = tfidf_vec.transform(X_train['text'].values.tolist())
    test_tfidf = tfidf_vec.transform(X_test['text'].values.tolist())
    return train_tfidf,test_tfidf

- analyzer='char' : 문자 기준 TF-IDF

In [52]:
def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

In [25]:
def do(X_train,X_test,Y_train):
    train_tfidf,test_tfidf = tfidfWords(X_train,X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_tfidf[dev_index], train_tfidf[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_tfidf)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.
    return pred_train, pred_full_test

In [54]:
pred_train,pred_test = do(X_train,X_test,Y_train)

Mean cv score :  0.9830868491998358


In [55]:
X_train["tfidf_chars_nb_cvec_0"] = pred_train[:,0]
X_train["tfidf_chars_nb_cvec_1"] = pred_train[:,1]
X_train["tfidf_chars_nb_cvec_2"] = pred_train[:,2]
X_train["tfidf_chars_nb_cvec_3"] = pred_train[:,3]
X_train["tfidf_chars_nb_cvec_4"] = pred_train[:,4]
X_test["tfidf_chars_nb_cvec_0"] = pred_test[:,0]
X_test["tfidf_chars_nb_cvec_1"] = pred_test[:,1]
X_test["tfidf_chars_nb_cvec_2"] = pred_test[:,2]
X_test["tfidf_chars_nb_cvec_3"] = pred_test[:,3]
X_test["tfidf_chars_nb_cvec_4"] = pred_test[:,4]

> TF-IDF + Naive Bayes로 1차 작가 문자 분류를 한 뒤, 그 예측 확률 5개를 새로운 feature로 추가하는 코드

| 방식          | 잘 잡는 것    |
| ----------- | --------- |
| word TF-IDF | 의미/어휘 선택  |
| char TF-IDF | 철자/문체/구두점 |


### Count

| 방식     | 특징       |
| ------ | -------- |
| Count  | 순수 빈도    |
| TF-IDF | 중요 단어 강조(희귀 단어 점수 높임) |


**< word - 단어단위 >**

In [28]:
def countWords(X_train, X_test):
    count_vec = CountVectorizer(stop_words='english', ngram_range=(1,2))
    count_vec.fit(X_train['text'].values.tolist())
    train_count = count_vec.transform(X_train['text'].values.tolist())
    test_count = count_vec.transform(X_test['text'].values.tolist())
    return train_count, test_count

def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

def do_count_MNB(X_train, X_test, Y_train):
    train_count, test_count = countWords(X_train, X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits = 8, shuffle=True, random_state=2020)
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_count[dev_index], train_count[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_count)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.
    return pred_train, pred_full_test

pred_train, pred_test = do_count_MNB(X_train, X_test, Y_train)

X_train["count_words_nb_cvec_0"] = pred_train[:,0]
X_train["count_words_nb_cvec_1"] = pred_train[:,1]
X_train["count_words_nb_cvec_2"] = pred_train[:,2]
X_train["count_words_nb_cvec_3"] = pred_train[:,3]
X_train["count_words_nb_cvec_4"] = pred_train[:,4]
X_test["count_words_nb_cvec_0"] = pred_test[:,0]
X_test["count_words_nb_cvec_1"] = pred_test[:,1]
X_test["count_words_nb_cvec_2"] = pred_test[:,2]
X_test["count_words_nb_cvec_3"] = pred_test[:,3]
X_test["count_words_nb_cvec_4"] = pred_test[:,4]

Mean cv score :  0.9921673388998574


**< word - 단어단위 >**

In [56]:
def countChars(X_train, X_test):
    count_vec = CountVectorizer(ngram_range=(3,4), analyzer='char')
    count_vec.fit(X_train['text'].values.tolist())
    train_count = count_vec.transform(X_train['text'].values.tolist())
    test_count = count_vec.transform(X_test['text'].values.tolist())
    return train_count,test_count

def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

def do_count_chars_MNB(X_train, X_test, Y_train):
    train_count,test_count=countChars(X_train, X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_count[dev_index], train_count[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_count)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.
    return pred_train, pred_full_test

pred_train, pred_test = do_count_chars_MNB(X_train, X_test, Y_train)

X_train["count_chars_nb_cvec_0"] = pred_train[:,0]
X_train["count_chars_nb_cvec_1"] = pred_train[:,1]
X_train["count_chars_nb_cvec_2"] = pred_train[:,2]
X_train["count_chars_nb_cvec_3"] = pred_train[:,3]
X_train["count_chars_nb_cvec_4"] = pred_train[:,4]
X_test["count_chars_nb_cvec_0"] = pred_test[:,0]
X_test["count_chars_nb_cvec_1"] = pred_test[:,1]
X_test["count_chars_nb_cvec_2"] = pred_test[:,2]
X_test["count_chars_nb_cvec_3"] = pred_test[:,3]
X_test["count_chars_nb_cvec_4"] = pred_test[:,4]

Mean cv score :  3.075051208358071


해설: 1. 단어 단위(Word)와 문자 단위(Char) 각각에 대해 TF-IDF 행렬을 만듭니다. (Char TF-IDF에서는 ngram_range=(3,4)로 지정해 글자 3~4개짜리 뭉치의 리듬을 포착했습니다.)
2. 이 거대한 TF-IDF 행렬을 메인 트리 모델(XGBoost)에 바로 넣지 않고, 계산이 극도로 빠른 다항 나이브 베이즈(MultinomialNB) 모델에 먼저 학습시킵니다.
3. 8-Fold 교차 검증(OOF; Out-of-Fold)을 돌려, 각 문장이 5명의 작가 각각에 속할 확률값(5개 컬럼)을 예측해 냅니다.
4. 이 5개의 확률 벡터를 tfidf_words_nb_cvec_0 ~ 4라는 새로운 피처(Meta Feature)로 데이터프레임에 붙여버립니다.

효과: 수만 개에 달하던 TF-IDF의 희소 행렬 정보가 "나이브 베이즈 모델이 보기에 이 문장은 작가 0번일 확률이 80%, 1번일 확률이 5%야"라는 정제된 5개의 압축 숫자로 변환된 것입니다.

---

In [58]:
#### 추가 부분

# Word TF-IDF 함수 다시 선언 - 중복 선언 방지
def tfidfWords_for_LR(X_train, X_test):
    # 단어 기준 설정
    tfidf_vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    train_tfidf = tfidf_vec.fit_transform(X_train['text'].values.tolist())
    test_tfidf = tfidf_vec.transform(X_test['text'].values.tolist())
    return train_tfidf, test_tfidf

# 로지스틱 회귀 메타 피처 생성 함수 정의
from sklearn.linear_model import LogisticRegression

def do_tfidf_LR_fixed(X_train, X_test, Y_train):
    # 위에서 만든 단어 TF-IDF 함수 호출
    train_tfidf, test_tfidf = tfidfWords_for_LR(X_train, X_test)

    pred_train = np.zeros([X_train.shape[0], 5])
    pred_full_test = np.zeros([X_test.shape[0], 5])

    # 동일한 8-Fold 설정
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)
    cv_scores = []

    print("[Word - Logistic Regression] 8-Fold 메타 피처 생성 시작...")

    for fold, (dev_index, val_index) in enumerate(kf.split(X_train)):
        dev_X, val_X = train_tfidf[dev_index], train_tfidf[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]

        # 로지스틱 회귀
        model = LogisticRegression(C=1.0, max_iter=300, solver='saga', n_jobs=-1, random_state=42)
        model.fit(dev_X, dev_y)

        pred_val_y = model.predict_proba(val_X)
        pred_test_y = model.predict_proba(test_tfidf)

        pred_train[val_index, :] = pred_val_y
        pred_full_test += pred_test_y

        fold_loss = metrics.log_loss(val_y, pred_val_y)
        cv_scores.append(fold_loss)
        print(f"  Fold {fold+1} LogLoss: {fold_loss:.4f}")

    print(f"⇒ Word LR Mean CV Score : {np.mean(cv_scores):.4f}\n")
    pred_full_test = pred_full_test / 8.

    return pred_train, pred_full_test

# 3. 실제로 실행하여 데이터프레임에 피처 결합하기
lr_word_train, lr_word_test = do_tfidf_LR_fixed(X_train, X_test, Y_train)

for i in range(5):
    X_train[f"tfidf_words_lr_cvec_{i}"] = lr_word_train[:, i]
    X_test[f"tfidf_words_lr_cvec_{i}"] = lr_word_test[:, i]

print("정상적으로 5개의 로지스틱 회귀 피처가 추가되었습니다! 데이터 모양을 확인하세요.")
print("X_train shape:", X_train.shape)

[Word - Logistic Regression] 8-Fold 메타 피처 생성 시작...
  Fold 1 LogLoss: 0.9099
  Fold 2 LogLoss: 0.9018
  Fold 3 LogLoss: 0.9171
  Fold 4 LogLoss: 0.9178
  Fold 5 LogLoss: 0.9157
  Fold 6 LogLoss: 0.9204
  Fold 7 LogLoss: 0.8951
  Fold 8 LogLoss: 0.9068
⇒ Word LR Mean CV Score : 0.9106

정상적으로 5개의 로지스틱 회귀 피처가 추가되었습니다! 데이터 모양을 확인하세요.
X_train shape: (54879, 144)


/tmp/ipykernel_3738/3365997015.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"tfidf_words_lr_cvec_{i}"] = lr_word_train[:, i]
/tmp/ipykernel_3738/3365997015.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[f"tfidf_words_lr_cvec_{i}"] = lr_word_test[:, i]
/tmp/ipykernel_3738/3365997015.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instea

점수 확인: 0.9106

보통 Naive Bayes 단독 점수(Word 기준 1.08, Char 기준 0.98)보다 로지스틱 회귀의 자체 점수가 더 낮게(좋게) 나오는 경우 有

## sentence embedding

- 단어를 의미 벡터(vector)로 바꾸고, 문장 전체를 하나의 벡터로 표현

GloVe 데이터 결합 - 규칙상 외부데이터 결합 불가

< GloVe >
- 단어 의미 자체를 숫자 벡터로 표현
- 단어를 숫자 벡터로 바꿔주는 사전
- 단어의 의미를 숫자 100개짜리 좌표로 저장해둔 파일
  - 의미가 비슷한 단어들이 비슷한 벡터를 가짐

In [30]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-05-15 08:30:18--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-05-15 08:30:19--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-15 08:30:19--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [59]:
wv = "./glove.6B.100d.txt"

In [60]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [61]:
from nltk import word_tokenize
from tqdm import tqdm

In [62]:
# GloVe 사전 불러오는 함수
def loadWordVecs():
    embeddings_index = {}
    f = open(wv)
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
    f.close()
    print('Found %s word vectors.' % len(embeddings_index))
    return embeddings_index

In [63]:
# 정규화 벡터 생성 함수 with GloVe
def sent2vec(embeddings_index, s):
    words = str(s).lower()
    words = word_tokenize(words)
    words = [w for w in words if not w in stopwords.words('english')]
    words = [w for w in words if w.isalpha()]  # 알파벳만 남김
    M = []
    for w in words:
        try:
            M.append(embeddings_index[w])  # 각 단어의 GloVe 벡터 가져오기
        except:
            continue
    M = np.array(M)
    v = M.sum(axis=0)  # 문장 벡터 = 단어 벡터들의 합
    if type(v) != np.ndarray:
        return np.zeros(100)
    return v / np.sqrt((v ** 2).sum())  # 정규화(벡터 길이 1로 맞춤)

In [64]:
def doGlove(x_train, x_test):
    embeddings_index = loadWordVecs()
    xtrain_glove = [sent2vec(embeddings_index,x) for x in tqdm(x_train)]
    xtest_glove = [sent2vec(embeddings_index,x) for x in tqdm(x_test)]
    xtrain_glove = np.array(xtrain_glove)
    xtest_glove = np.array(xtest_glove)
    return xtrain_glove, xtest_glove, embeddings_index

In [65]:
glove_vecs_train, glove_vecs_test, embeddings_index = doGlove(X_train['text'], X_test['text'])
X_train[['sent_vec_'+str(i) for i in range(100)]] = pd.DataFrame(glove_vecs_train.tolist())
X_test[['sent_vec_'+str(i) for i in range(100)]] = pd.DataFrame(glove_vecs_test.tolist())

Found 400000 word vectors.


100%|██████████| 19617/19617 [02:09<00:00, 151.92it/s]


문장에 포함된 단어들의 100차원 벡터를 모두 더한 뒤 정규화(v / np.sqrt((v  2).sum()))하여, 그 문장이 가진 고유한 서사적 분위기나 의미적 맥락을 100개의 컬럼(sent_vec_0 ~ 99)으로 추가

## DeepLearning

# Modeling

In [66]:
from sklearn.metrics import log_loss, accuracy_score
from xgboost import XGBClassifier

## XGBoost

- 숫자형 feature만 선택
- 사용 O: punc, stop_word, num_words, NB 확률 feature, GloVe sent_vec
- 사용 X: text, words, id

In [67]:
X_train.select_dtypes(include='object').columns

Index(['text', 'words'], dtype='object')

In [68]:
drop_cols = ['text', 'words', 'author']

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
X_xgb = X_train_xgb.drop(columns=drop_cols, errors='ignore')
X_test_xgb = X_test_xgb.drop(columns=drop_cols, errors='ignore')
y_xgb = Y_train.copy()

텍스트 원문과 리스트 타입을 제외한 오직 '숫자형 피처'만 선택  


In [69]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_xgb,
    y_xgb,
    test_size=0.2,
    random_state=42,
    stratify=y_xgb
)

xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_tr, y_tr)

val_pred = xgb.predict_proba(X_val)

print("log_loss:", log_loss(y_val, val_pred))
print("accuracy:", accuracy_score(y_val, val_pred.argmax(axis=1)))

log_loss: 0.45466769746603436
accuracy: 0.8333637026239067


In [77]:
xgb.fit(X_xgb, y_xgb)
xgb_pred = xgb.predict_proba(X_test_xgb)

## LGBM

In [70]:
from lightgbm import LGBMClassifier

In [71]:
drop_cols = ['text', 'words', 'author']

X_train_lgbm = X_train.copy()
X_test_lgbm = X_test.copy()
X_lgbm = X_train_lgbm.drop(columns=drop_cols, errors='ignore')
X_test_lgbm = X_test_lgbm.drop(columns=drop_cols, errors='ignore')
y_lgbm = Y_train.copy()

In [72]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_lgbm,
    y_lgbm,
    test_size=0.2,
    random_state=42,
    stratify=y_lgbm
)

lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=5,
    n_estimators=500,
    learning_rate=0.03,
    random_state=42
)

lgbm.fit(X_tr, y_tr)

val_pred = lgbm.predict_proba(X_val)

print("log_loss:", log_loss(y_val, val_pred))
print("accuracy:", accuracy_score(y_val, val_pred.argmax(axis=1)))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.110414 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 35053
[LightGBM] [Info] Number of data points in the train set: 43903, number of used features: 141
[LightGBM] [Info] Start training from score -1.422261
[LightGBM] [Info] Start training from score -2.027925
[LightGBM] [Info] Start training from score -1.558116
[LightGBM] [Info] Start training from score -1.292918
[LightGBM] [Info] Start training from score -1.950362
log_loss: 0.45298624167184076
accuracy: 0.8360969387755102


In [73]:
final_lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=5,
    n_estimators=500,
    learning_rate=0.03,
    random_state=42
)

final_lgbm.fit(X_lgbm, y_lgbm)

lgbm_pred = final_lgbm.predict_proba(X_test_lgbm)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.117617 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 35061
[LightGBM] [Info] Number of data points in the train set: 54879, number of used features: 141
[LightGBM] [Info] Start training from score -1.422266
[LightGBM] [Info] Start training from score -2.027999
[LightGBM] [Info] Start training from score -1.558099
[LightGBM] [Info] Start training from score -1.292889
[LightGBM] [Info] Start training from score -1.950366


XGBoost 및 LightGBM 독립 학습

---

# 디벨롭 제안

## 1: 메타 피처(압축기) 고도화

MNB 피처 외에 다른 모델을 압축기로 추가
- 로지스틱 회귀(Logistic Regression) 메타 피처 추가: 텍스트 분류에서는 선형 모델인 로지스틱 회귀가 나이브 베이즈보다 개별 성능이 더 뛰어난 경우가 많음
- 기존 나이브 베이즈 확률 5개 컬럼에 더해, 로지스틱 회귀가 예측한 작가별 확률 5개 컬럼을 피처로 추가(lr_words_pred_0~4)

[비교]
- XGBoost 단독: 0.4534
- LGBM 단독: 0.4522

피처 추가 후 성능
- XGBoost: 0.4546
- LGBM: 0.4529

In [78]:
# 두 모델의 결과 앙상블 (Soft Voting)
# 우선 가장 안정적인 5:5 비율로 두 모델의 확률값을 평균
final_ensemble_pred = (xgb_pred + lgbm_pred) / 2.0

In [79]:
sub.iloc[:, 1:] = final_ensemble_pred

output_filename = 'submission_v2_lr_features.csv'
sub.to_csv(output_filename, index=False)

/tmp/ipykernel_3738/2698335858.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2.50758314e-02 1.88564565e-02 9.98180602e-01 ... 1.00893485e-03
 2.95252758e-04 9.45687853e-01]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sub.iloc[:, 1:] = final_ensemble_pred
/tmp/ipykernel_3738/2698335858.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[5.67679702e-01 9.55133260e-01 8.81986367e-04 ... 9.98505758e-01
 9.97279366e-01 5.16720759e-03]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sub.iloc[:, 1:] = final_ensemble_pred
/tmp/ipykernel_3738/2698335858.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.19819952 0.00110593 0.00023106 ... 0.00019822 0.00165697 0.02746115]' has dtype in

# 2: Optuna 파라미터 최적

In [81]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.0 MB/s eta 0:00:00


In [83]:
import optuna
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 5,
        'n_estimators': 700,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    # 튜닝 속도를 위해 3-Fold 셋업 유지
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X_lgbm, y_lgbm):
        X_tr, X_val = X_lgbm.iloc[train_idx], X_lgbm.iloc[val_idx]
        y_tr, y_val = y_lgbm[train_idx], y_lgbm[val_idx]

        model = LGBMClassifier(**params)

        # 에러가 나던 Optuna 콜백 대신, LightGBM 자체 조기 종료(early_stopping) 활용
        # 30번 동안 점수 개선이 없으면 해당 트리 생성을 중단하여 시간을 절약
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
        )

        preds = model.predict_proba(X_val)
        cv_scores.append(log_loss(y_val, preds))

    return np.mean(cv_scores)

# Optuna 최적화 실행
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20) # 20회 실험

print("\n" + "="*50)
print(f"최적의 LogLoss 점수: {study.best_value:.4f}")
print("최적의 하이퍼파라미터 조합:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print("="*50)

[I 2026-05-15 09:23:22,710] A new study created in memory with name: no-name-8fc82e11-cb64-4ca5-bd2b-c64aebaf8391
[I 2026-05-15 09:26:15,697] Trial 0 finished with value: 0.4578791011647414 and parameters: {'learning_rate': 0.04170272925451585, 'num_leaves': 58, 'max_depth': 5, 'min_child_samples': 24, 'subsample': 0.6918225111811602, 'colsample_bytree': 0.6095805615745072, 'reg_alpha': 0.006207129762822163, 'reg_lambda': 6.616694517177458}. Best is trial 0 with value: 0.4578791011647414.
[I 2026-05-15 09:28:56,823] Trial 1 finished with value: 0.4653617623472921 and parameters: {'learning_rate': 0.04545482729634783, 'num_leaves': 78, 'max_depth': 9, 'min_child_samples': 35, 'subsample': 0.6962957379771207, 'colsample_bytree': 0.8836398981536571, 'reg_alpha': 0.8179049216642399, 'reg_lambda': 0.04154294854427222}. Best is trial 0 with value: 0.4578791011647414.
[I 2026-05-15 09:34:31,048] Trial 2 finished with value: 0.4678515631157432 and parameters: {'learning_rate': 0.02122438028246


최적의 LogLoss 점수: 0.4579
최적의 하이퍼파라미터 조합:
  learning_rate: 0.04170272925451585
  num_leaves: 58
  max_depth: 5
  min_child_samples: 24
  subsample: 0.6918225111811602
  colsample_bytree: 0.6095805615745072
  reg_alpha: 0.006207129762822163
  reg_lambda: 6.616694517177458


성능: 0.4579

In [84]:
import pandas as pd
from lightgbm import LGBMClassifier

# Optuna 최적 하이퍼파라미터 반영
best_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'n_estimators': 700,                             # 조금 더 조밀한 학습을 위해 에스티메이터 유지
    'learning_rate': 0.04170272925451585,
    'num_leaves': 58,
    'max_depth': 5,
    'min_child_samples': 24,
    'subsample': 0.6918225111811602,
    'colsample_bytree': 0.6095805615745072,
    'reg_alpha': 0.006207129762822163,
    'reg_lambda': 6.616694517177458,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}


final_optuna_lgbm = LGBMClassifier(**best_params)

# 로지스틱 회귀 피처까지 전부 포함된 전체 데이터셋(X_lgbm)으로 학습
final_optuna_lgbm.fit(X_lgbm, y_lgbm)

# 테스트 데이터에 대한 작가별 확률 예측
optuna_lgbm_pred = final_optuna_lgbm.predict_proba(X_test_lgbm)

In [85]:
# 제출 파일 저장
sub_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/sample_submission.csv'
sub = pd.read_csv(sub_path)

sub.iloc[:, 1:] = optuna_lgbm_pred
output_filename = 'submission_v3_optuna_lgbm.csv'
sub.to_csv(output_filename, index=False)

리더보드 점수: 0.2309530018  
더 점수 개선됨.

## 3: Optuna 최적화 LGBM + XGB 앙상블

기존에 뽑아두었던 xgb_pred(40%)와 방금 구한 optuna_lgbm_pred(60%)를 결합  
LGBM에 약간 가중치 더 줌.  
단독 모델 오차 줄여보기

In [86]:
final_weight_blend = (xgb_pred * 0.4) + (optuna_lgbm_pred * 0.6)

In [88]:
# 제출파일 저장
sub_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/sample_submission.csv'
sub = pd.read_csv(sub_path)

sub.iloc[:, 1:] = final_weight_blend
output_filename = 'submission_v4_final_blend_64.csv'
sub.to_csv(output_filename, index=False)

리더보드 점수: 0.2335651229 (다소 점수 하락)

## 4: 앙상블 가중치 조정
LGBM:XGB = 8:2로 변경  
XGB의 노이즈 효과 줄여보기

In [89]:
final_weight_blend_82 = (xgb_pred * 0.2) + (optuna_lgbm_pred * 0.8)

In [90]:
# 제출파일 저장
sub_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/sample_submission.csv'
sub = pd.read_csv(sub_path)

sub.iloc[:, 1:] = final_weight_blend_82
output_filename = 'submission_v5_blend_82.csv'
sub.to_csv(output_filename, index=False)

리더보드 점수: 0.2318344058  
가중치 조정 전보단 낫지만 단독보다는 점수 낮음.

XGB 파라미터 개선 필요?

## 5: 최적화 XGB + 최적화 LGBM 앙상블(5:5)
XGB도 Optuna 파라미터 최적화 후 앙상블

In [93]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def objective_xgb(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 5,
        'n_estimators': 600,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
        'max_depth': trial.suggest_int('max_depth', 4, 8),                  # 너무 깊어지지 않게 제한
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),    # 과적합 방지 핵심
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist', # 코랩 환경에서 대폭적인 속도 향상을 위한 옵션
        'early_stopping_rounds': 30
    }

    # 3-Fold(시간 절약)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X_lgbm, y_lgbm):
        X_tr, X_val = X_lgbm.iloc[train_idx], X_lgbm.iloc[val_idx]
        y_tr, y_val = y_lgbm[train_idx], y_lgbm[val_idx]

        model = XGBClassifier(**params)

        # 30번 동안 개선 없으면 조기 종료하여 서치 속도를 극대화
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict_proba(X_val)
        cv_scores.append(log_loss(y_val, preds))

    return np.mean(cv_scores)

In [94]:
# Optuna 서치
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=15)


print(f"XGBoost LogLoss 점수: {study_xgb.best_value:.4f}")
print("최적의 하이퍼파라미터 조합:")
for key, value in study_xgb.best_params.items():
    print(f"  {key}: {value}")
print("="*50)

[I 2026-05-15 11:20:36,121] A new study created in memory with name: no-name-cd73fffd-32ea-4e46-a730-a94d9454f70f
[I 2026-05-15 11:25:40,037] Trial 0 finished with value: 0.46137834186410037 and parameters: {'learning_rate': 0.027182492309410528, 'max_depth': 5, 'min_child_weight': 3, 'subsample': 0.8864690412465197, 'colsample_bytree': 0.6871693904445337, 'reg_alpha': 0.30313654165863996, 'reg_lambda': 0.06350542858788037}. Best is trial 0 with value: 0.46137834186410037.
[I 2026-05-15 11:32:18,057] Trial 1 finished with value: 0.4593065540188694 and parameters: {'learning_rate': 0.04483255764497944, 'max_depth': 6, 'min_child_weight': 8, 'subsample': 0.6749137807419724, 'colsample_bytree': 0.8543259961093141, 'reg_alpha': 4.439804842060363, 'reg_lambda': 0.002016692121884858}. Best is trial 1 with value: 0.4593065540188694.
[I 2026-05-15 11:43:00,095] Trial 2 finished with value: 0.4596376513820384 and parameters: {'learning_rate': 0.021825985139659415, 'max_depth': 8, 'min_child_wei

XGBoost LogLoss 점수: 0.4570
최적의 하이퍼파라미터 조합:
  learning_rate: 0.04927134571682068
  max_depth: 5
  min_child_weight: 5
  subsample: 0.6849204779825756
  colsample_bytree: 0.7412846646468872
  reg_alpha: 1.4603521710855918
  reg_lambda: 0.0062597669289390645


In [95]:
import pandas as pd
from xgboost import XGBClassifier

# Optuna XGBoost 최적 하이퍼파라미터 반영
best_xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 5,
    'n_estimators': 600,
    'learning_rate': 0.04927134571682068,
    'max_depth': 5,
    'min_child_weight': 5,
    'subsample': 0.6849204779825756,
    'colsample_bytree': 0.7412846646468872,
    'reg_alpha': 1.4603521710855918,
    'reg_lambda': 0.0062597669289390645,
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist'
}

final_optuna_xgb = XGBClassifier(**best_xgb_params)
final_optuna_xgb.fit(X_lgbm, y_lgbm)
optuna_xgb_pred = final_optuna_xgb.predict_proba(X_test_lgbm)

In [96]:
# 5:5 가중치(LGBM 50% + 튜닝 XGB 50%)
master_blend = (optuna_lgbm_pred * 0.5) + (optuna_xgb_pred * 0.5)

In [97]:
# 제출파일 저장
sub_path = '/content/drive/MyDrive/EWHA/ESAA/26-1 OB/OB 프로젝트(2)/data/sample_submission.csv'
sub = pd.read_csv(sub_path)

sub.iloc[:, 1:] = master_blend
output_filename = 'submission_v6_master_blend_55.csv'
sub.to_csv(output_filename, index=False)

리더보드 점수: 0.22